In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

import torch
import torch.nn as nn
from collections import deque
import sentencepiece as spm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

relu = nn.ReLU()

def relu_derivative(x):
    return (x > 0).float()

def positional_encoding(max_seq_len, d_model):
    PE = torch.zeros(max_seq_len, d_model)
    pos = torch.arange(0, max_seq_len).unsqueeze(1)
    i = torch.arange(0, d_model, 2)
    div = 10000 ** (i / d_model)
    PE[:, 0::2] = torch.sin(pos / div)
    PE[:, 1::2] = torch.cos(pos / div)
    return PE

## Model Architecture

In [ ]:
class Attention_Block(nn.Module):
  def __init__(self, d_model:int, num_heads:int):
    super().__init__()
    self.num_heads = num_heads
    self.input = None
    self.Exp = None
    self.d_k = d_model // num_heads
    self.d_model = d_model
    self.Q_M = nn.Parameter(torch.randn(num_heads, d_model, self.d_k))
    self.K_M = nn.Parameter(torch.randn(num_heads, d_model, self.d_k))
    self.V_M = nn.Parameter(torch.randn(num_heads, d_model, self.d_k))
    self.W_O = nn.Parameter(torch.randn(d_model, d_model))
    self.Q = None
    self.K = None
    self.V = None
    self.scores = None
    self.A = None
    self.deltaE_heads = None
    self.deltaE_cat = None

  def forward_pass(self, E, mask=False):
    n = E.shape[0]
    self.inputs = E
    self.Exp = self.inputs.unsqueeze(0).expand(self.num_heads, -1, -1)
    self.Q = torch.bmm(self.Exp, self.Q_M)
    self.K = torch.bmm(self.Exp, self.K_M)
    self.V = torch.bmm(self.Exp, self.V_M)

    self.scores = torch.bmm(self.Q, self.K.transpose(-2, -1)) / (self.d_k) ** 0.5
    if mask:
      m = torch.triu(torch.ones(n, n), diagonal=1).bool().to(self.Q_M.device)
      self.scores = self.scores.masked_fill(m, float('-inf')).to(device)

    self.A = torch.softmax(self.scores, dim=-1)
    self.deltaE_heads = torch.bmm(self.A, self.V)
    self.deltaE_cat = self.deltaE_heads.transpose(0, 1).reshape(n, -1)

    return self.deltaE_cat @ self.W_O

  def backpropagation(self, i_g):
    n = self.inputs.shape[0]

    dW_O = self.deltaE_cat.T @ i_g
    d_deltaE_cat = i_g @ self.W_O.T

    d_deltaE_heads = d_deltaE_cat.reshape(n, self.num_heads, self.d_k).transpose(0, 1)

    dA = torch.bmm(d_deltaE_heads, self.V.transpose(-2, -1))
    dV = torch.bmm(self.A.transpose(-2, -1), d_deltaE_heads)

    # softmax jacobian
    dS = self.A * (dA - (dA * self.A).sum(dim=-1, keepdim=True))

    dQ = torch.bmm(dS, self.K) / self.d_k ** 0.5
    dK = torch.bmm(dS.transpose(-2, -1), self.Q) / self.d_k ** 0.5

    dW_Q = torch.bmm(self.Exp.transpose(-2, -1), dQ)
    dW_K = torch.bmm(self.Exp.transpose(-2, -1), dK)
    dW_V = torch.bmm(self.Exp.transpose(-2, -1), dV)

    dX = (torch.bmm(dQ, self.Q_M.transpose(-2, -1)) +
          torch.bmm(dK, self.K_M.transpose(-2, -1)) +
          torch.bmm(dV, self.V_M.transpose(-2, -1))).sum(dim=0)

    return dW_Q, dW_K, dW_V, dW_O, dX

In [ ]:
class Layer(nn.Module):
  def __init__(self, input, prev):
    super().__init__()
    self.weights = nn.Parameter(torch.randn(prev, input))
    self.bias = nn.Parameter(torch.randn(input))
    self.activations = None
    self.z = None
    self.inputs = input
    self.prev = prev


class MLP(nn.Module):
  def __init__(self, num_layers:int, layers:list):
    super().__init__()
    self.num = num_layers
    self.layers = nn.ModuleList()
    self.l = layers
    self.input = None
    for i in range(1, self.num + 1):
      self.layers.append(Layer(layers[i], layers[i - 1]))

  def forward_pass(self, input):
    self.input = input
    current = input
    for i in range(0, self.num - 1):
        self.layers[i].z = (current @ self.layers[i].weights) + self.layers[i].bias
        self.layers[i].activations = relu(self.layers[i].z)
        current = self.layers[i].activations

    last = self.num - 1
    self.layers[last].z = current @ self.layers[last].weights + self.layers[last].bias
    self.layers[last].activations = self.layers[last].z

    return self.layers[last].activations

  def backpropagation(self, i_g):
    dW = deque()
    db = deque()
    delta = i_g

    for i in range(self.num - 1, -1, -1):
        if i != self.num - 1:
            delta = delta @ self.layers[i+1].weights.T * relu_derivative(self.layers[i].z)
        db.appendleft(delta.sum(dim=0))
        prev_act = self.input if i == 0 else self.layers[i-1].activations
        dW.appendleft(prev_act.T @ delta)

    d_input = delta @ self.layers[0].weights.T
    return dW, db, d_input


class LayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.input = None
        self.mu = None
        self.var = None
        self.x_hat = None

    def forward_pass(self, E):
        self.input = E
        self.mu = E.sum(dim=-1, keepdim=True) / self.d_model
        self.var = ((E - self.mu) ** 2).sum(dim=-1, keepdim=True) / self.d_model
        self.x_hat = (E - self.mu) / (self.var + 1e-5) ** 0.5
        return self.gamma * self.x_hat + self.beta

    def backpropagation(self, i_g):
        sigma = (self.var + 1e-5) ** 0.5

        d_gamma = (i_g * self.x_hat).sum(dim=0)
        d_beta = i_g.sum(dim=0)

        dx_hat = i_g * self.gamma

        # quotient rule accounting for mu and sigma both depending on X
        dX = (self.gamma / sigma) * (
            dx_hat
            - dx_hat.sum(dim=-1, keepdim=True) / self.d_model
            - self.x_hat * (dx_hat * self.x_hat).sum(dim=-1, keepdim=True) / self.d_model
        )

        return d_gamma, d_beta, dX

In [ ]:
class Transformer(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.attention = Attention_Block(d_model, num_heads).to(device)
        self.mlp = MLP(3, [d_model, 4*d_model, 4*d_model, d_model]).to(device)
        self.norm1 = LayerNorm(d_model).to(device)
        self.norm2 = LayerNorm(d_model).to(device)
        self.input = None
        self.norm1_out = None
        self.attn_out = None
        self.norm2_out = None
        self.mlp_out = None

    def forward_pass(self, E, mask=False):
        self.input = E

        self.norm1_out = self.norm1.forward_pass(E)
        self.attn_out = self.attention.forward_pass(self.norm1_out, mask)
        E = E + self.attn_out

        self.norm2_out = self.norm2.forward_pass(E)
        self.mlp_out = self.mlp.forward_pass(self.norm2_out)
        E = E + self.mlp_out

        return E

    def backpropagation(self, i_g):
        # residual: gradient copies to both branches, then sums when they rejoin
        d_mlp_out = i_g
        d_E_mid = i_g

        dW_mlp, db_mlp, d_norm2_out = self.mlp.backpropagation(d_mlp_out)
        d_gamma2, d_beta2, d_norm2_in = self.norm2.backpropagation(d_norm2_out)

        d_E_mid = d_E_mid + d_norm2_in

        d_attn_out = d_E_mid
        d_input = d_E_mid

        dW_Q, dW_K, dW_V, dW_O, d_norm1_out = self.attention.backpropagation(d_attn_out)
        d_gamma1, d_beta1, d_norm1_in = self.norm1.backpropagation(d_norm1_out)

        d_input = d_input + d_norm1_in

        grads = {
            'dW_Q': dW_Q, 'dW_K': dW_K, 'dW_V': dW_V, 'dW_O': dW_O,
            'dW_mlp': dW_mlp, 'db_mlp': db_mlp,
            'd_gamma1': d_gamma1, 'd_beta1': d_beta1,
            'd_gamma2': d_gamma2, 'd_beta2': d_beta2,
        }

        return grads, d_input


class LLM(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_blocks, max_seq_len):
        super().__init__()
        self.embedding = nn.Parameter(torch.randn(vocab_size, d_model))
        self.pos_encoding = positional_encoding(max_seq_len, d_model).to(device)
        self.blocks = nn.ModuleList([
            Transformer(d_model, num_heads)
            for _ in range(num_blocks)
        ])
        self.unembedding = nn.Parameter(torch.randn(d_model, vocab_size))
        self.token_ids = None
        self.E = None
        self.logits = None
        self.probs = None
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.embedding,   mean=0, std=0.02)
        nn.init.normal_(self.unembedding, mean=0, std=0.02)

        for block in self.blocks:
            nn.init.normal_(block.attention.Q_M, mean=0, std=0.02)
            nn.init.normal_(block.attention.K_M, mean=0, std=0.02)
            nn.init.normal_(block.attention.V_M, mean=0, std=0.02)
            nn.init.normal_(block.attention.W_O, mean=0, std=0.02)

            for layer in block.mlp.layers:
                nn.init.normal_(layer.weights, mean=0, std=0.02)
                nn.init.zeros_(layer.bias)

    def forward_pass(self, token_ids, mask=False):
        self.token_ids = token_ids
        self.E = self.embedding[token_ids] + self.pos_encoding[:len(token_ids)]
        for block in self.blocks:
            self.E = block.forward_pass(self.E, mask)
        self.logits = self.E @ self.unembedding        # (n, vocab_size)
        self.probs = torch.softmax(self.logits, dim=-1)
        return self.probs

    def evaluate(self, token_ids):
        # no mask, returns only last token probs for generation
        self.token_ids = token_ids
        self.E = self.embedding[token_ids] + self.pos_encoding[:len(token_ids)]
        for block in self.blocks:
            self.E = block.forward_pass(self.E, mask=False)
        logits = self.E[-1] @ self.unembedding
        probs = torch.softmax(logits, dim=-1)
        return probs

    def loss(self, target_ids):
        target_ids = target_ids[:-1]
        n = len(target_ids)
        p_correct = self.probs[:-1][torch.arange(n, device=self.probs.device), target_ids]
        return -torch.log(p_correct + 1e-9).mean()

    def backpropagation(self, target_ids):
        all_grads = []
        target_ids = target_ids[:-1]
        n = len(target_ids)
        dev = self.probs.device

        d_logits = torch.zeros_like(self.probs)
        d_logits[:-1] = self.probs[:-1].clone()
        d_logits[:-1][torch.arange(n, device=dev), target_ids] -= 1.0
        d_logits[:-1] /= n

        dW_unembed = self.E.T @ d_logits
        d_E = d_logits @ self.unembedding.T

        for block in reversed(self.blocks):
            grads, d_E = block.backpropagation(d_E)
            all_grads.append(grads)

        d_embedding = torch.zeros_like(self.embedding)
        for t, token_id in enumerate(self.token_ids):
            d_embedding[token_id] += d_E[t]

        return all_grads, dW_unembed, d_embedding

In [ ]:
class AdamW:
    def __init__(self, params, lr=1e-4, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.01):
        self.params = list(params)
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.weight_decay = weight_decay
        self.t = 0

        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]

    def flatten_grads(self, all_grads, dW_unembed, d_embedding):
        flat = []
        flat.append(d_embedding)   # embedding
        flat.append(dW_unembed)    # unembedding

        for block_grads in reversed(all_grads):
            flat.append(block_grads['dW_Q'])
            flat.append(block_grads['dW_K'])
            flat.append(block_grads['dW_V'])
            flat.append(block_grads['dW_O'])

            dW_mlp = list(block_grads['dW_mlp'])
            db_mlp = list(block_grads['db_mlp'])
            for dW, db in zip(dW_mlp, db_mlp):
                flat.append(dW)
                flat.append(db)

            flat.append(block_grads['d_gamma1'])
            flat.append(block_grads['d_beta1'])
            flat.append(block_grads['d_gamma2'])
            flat.append(block_grads['d_beta2'])

        return flat

    def step(self, all_grads, dW_unembed, d_embedding):
        self.t += 1
        grads = self.flatten_grads(all_grads, dW_unembed, d_embedding)

        bc1 = 1 - self.beta1 ** self.t
        bc2 = 1 - self.beta2 ** self.t

        for i, (p, g) in enumerate(zip(self.params, grads)):
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * g ** 2

            m_hat = self.m[i] / bc1
            v_hat = self.v[i] / bc2

            p.data -= self.lr * m_hat / (v_hat.sqrt() + self.eps)
            p.data -= self.lr * self.weight_decay * p.data

In [ ]:
class Tokenizer:
    def __init__(self, model_path):
        self.sp = spm.SentencePieceProcessor()
        self.sp.load(model_path)
        self.vocab_size = self.sp.get_piece_size()

    def encode(self, text):
        return self.sp.encode(text)

    def decode(self, ids):
        return self.sp.decode(ids)


class AIAgent:
    def __init__(self, vocab_size, d_model, num_heads, num_blocks, max_seq_len):
        self.max_seq_len = max_seq_len
        self.tokenizer = None
        self.llm = LLM(vocab_size, d_model, num_heads, num_blocks, max_seq_len).to(device)
        self.optimizer = AdamW(self.llm.parameters(), lr=1e-4, weight_decay=0.01)

    def generate(self, prompt, num_tokens=100, temperature=1.0):
        assert self.tokenizer is not None, "Assign tokenizer first: agent.tokenizer = tokenizer"

        token_ids = self.tokenizer.encode(prompt)
        token_ids = token_ids[-self.max_seq_len:]
        generated = list(token_ids)
        prompt_len = len(token_ids)

        with torch.no_grad():
            for _ in range(num_tokens):
                input_tensor = torch.tensor(generated[-self.max_seq_len:], dtype=torch.long).to(device)
                probs = self.llm.evaluate(input_tensor)

                probs = probs ** (1.0 / temperature)
                probs = probs / probs.sum()

                next_token = torch.multinomial(probs, num_samples=1).item()
                generated.append(next_token)

        return self.tokenizer.decode(generated[prompt_len:])

    def train(self, text, epochs=1, seq_len=None, log_every=100, save_path=None):
        assert self.tokenizer is not None, "Assign tokenizer first: agent.tokenizer = tokenizer"

        if seq_len is None:
            seq_len = self.max_seq_len

        all_tokens = self.tokenizer.encode(text)
        total_tokens = len(all_tokens)
        print(f"total tokens: {total_tokens}")
        print(f"sequence length: {seq_len}")
        print(f"steps per epoch: {total_tokens // seq_len}")

        global_step = 0

        for epoch in range(epochs):
            epoch_loss = 0
            num_steps  = 0

            for start in range(0, total_tokens - seq_len, seq_len):
                chunk = all_tokens[start : start + seq_len + 1]

                if len(chunk) < 2:
                    continue

                inputs  = chunk[:-1]
                targets = chunk[1:]

                input_tensor  = torch.tensor(inputs,  dtype=torch.long).to(device)
                target_tensor = torch.tensor(targets, dtype=torch.long).to(device)

                with torch.no_grad():
                    self.llm.forward_pass(input_tensor, mask=True)
                    loss = self.llm.loss(target_tensor)
                    all_grads, dW_unembed, d_embedding = self.llm.backpropagation(target_tensor)
                    self.optimizer.step(all_grads, dW_unembed, d_embedding)

                epoch_loss  += loss.item()
                num_steps   += 1
                global_step += 1

                if global_step % 50 == 0:
                    torch.cuda.empty_cache()

                if global_step % log_every == 0:
                    avg_loss = epoch_loss / num_steps
                    print(f"epoch {epoch+1} | step {global_step} | loss {avg_loss:.4f}")

            avg_loss = epoch_loss / max(num_steps, 1)
            print(f"--- epoch {epoch+1} complete | avg loss {avg_loss:.4f} ---")

            if save_path:
                self.save(save_path)
                print(f"checkpoint saved")

    def save(self, path):
        torch.save({
            'model': self.llm.state_dict(),
            'optimizer_m': self.optimizer.m,
            'optimizer_v': self.optimizer.v,
            'optimizer_t': self.optimizer.t,
        }, path)
        print(f"Saved to {path}")

    def load(self, path):
        checkpoint = torch.load(path, map_location=device)
        self.llm.load_state_dict(checkpoint['model'])
        self.optimizer.m = checkpoint['optimizer_m']
        self.optimizer.v = checkpoint['optimizer_v']
        self.optimizer.t = checkpoint['optimizer_t']
        print(f"Loaded from {path}")


print("All classes loaded successfully")

## Load Corpus & Tokenizer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

with open('/content/drive/MyDrive/quantum_corpus.txt', 'r', encoding='utf-8') as f:
    corpus = f.read()

print(f"corpus loaded: {len(corpus)} characters")

## (Optional) Re-scrape Corpus
Only run this if you need to rebuild the corpus from scratch.

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import re

def scrape_wikipedia(topic):
    url = f"https://en.wikipedia.org/wiki/{topic.replace(' ', '_')}"
    headers = {'User-Agent': 'Mozilla/5.0 (compatible; research bot; your@email.com)'}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')

    content = soup.find('div', {'id': 'mw-content-text'})
    if not content:
        return ""

    paragraphs = content.find_all('p')
    text = ' '.join(p.get_text() for p in paragraphs)
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

quantum_topics = [
    "Quantum_mechanics", "Wave_function", "Schrödinger_equation",
    "Quantum_entanglement", "Heisenberg_uncertainty_principle",
    "Quantum_superposition", "Planck_constant", "Double-slit_experiment",
    "Quantum_tunnelling", "Bra-ket_notation",
]
greetings_topics = ["Greeting", "Small_talk"]

corpus = ""
for topic in quantum_topics + greetings_topics:
    print(f"  scraping: {topic}")
    corpus += scrape_wikipedia(topic) + "\n\n"
    time.sleep(1)

print(f"corpus length: {len(corpus)} characters")

with open('/content/drive/MyDrive/quantum_corpus.txt', 'w', encoding='utf-8') as f:
    f.write(corpus)
print("Saved to Google Drive")

## (Optional) Re-train Tokenizer
Only run this if you need to rebuild the sentencepiece tokenizer from scratch.

In [ ]:
import re

clean_corpus = re.sub(r'([.!?])\s+', r'\1\n', corpus)

with open('/content/corpus_temp.txt', 'w', encoding='utf-8') as f:
    f.write(clean_corpus)

spm.SentencePieceTrainer.train(
    input='/content/corpus_temp.txt',
    model_prefix='/content/drive/MyDrive/quantum_spm',
    vocab_size=1000,
    character_coverage=0.9995,
    model_type='bpe',
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
    input_sentence_size=1000000,
    shuffle_input_sentence=True,
    minloglevel=0,
)
print("tokenizer trained")

## Initialize Agent

In [ ]:
tokenizer = Tokenizer('/content/drive/MyDrive/quantum_spm.model')
print(f"vocab size: {tokenizer.vocab_size}")

agent = AIAgent(
    vocab_size  = tokenizer.vocab_size,
    d_model     = 256,
    num_heads   = 8,
    num_blocks  = 4,
    max_seq_len = 512
)
agent.tokenizer = tokenizer
print("Agent ready")

## Resume from Checkpoint
Run this instead of training from scratch if you have a saved checkpoint.

In [ ]:
agent.load('/content/drive/MyDrive/quantum_llm_v3.pt')

## Train

In [ ]:
agent.train(
    text       = corpus,
    epochs     = 30,
    seq_len    = 512,
    log_every  = 100,
    save_path  = '/content/drive/MyDrive/quantum_llm_v3.pt'
)

## Generate

In [ ]:
print(agent.generate("What is quantum mechanics", num_tokens=300, temperature=0.8))
print("---")
print(agent.generate("the wavefunction", num_tokens=300, temperature=0.8))
print("---")
print(agent.generate("hello how are you", num_tokens=300, temperature=0.8))